# Week 2 – Lasso, Ridge, and Elastic Net Regression

## Objective

This notebook applies regularized linear regression methods to the NBA game total prediction project. The goal is to compare ordinary linear regression with Ridge, Lasso, and Elastic Net regression and evaluate whether regularization improves predictive performance or model interpretability.

The target variable is `actual_total`, representing the total points scored in an NBA game.

Predictors:
- Sportsbook total
- Point spread
- Home moneyline
- Away moneyline

Models are evaluated using R², MAE, and RMSE.

In [21]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    root_mean_squared_error
)
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, KFold


In [14]:
merged = pd.read_csv("../data/processed/merged_games_betting.csv")
print("Dataset shape:", merged.shape)
print("Zero-score games:", (merged["actual_total"] == 0).sum())

Dataset shape: (21965, 54)
Zero-score games: 0


/var/folders/fj/fkly19253s58fj8xwtwph5cw0000gn/T/ipykernel_94530/3162278193.py:1: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  merged = pd.read_csv("../data/processed/merged_games_betting.csv")


In [15]:
features = [
    "total",
    "spread",
    "moneyline_home",
    "moneyline_away"
]

target = "actual_total"
model_df = merged[features + [target]].dropna()

X = model_df[features]
y = model_df[target]

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

## Feature Scaling

Ridge, Lasso, and Elastic Net apply penalties to model coefficients. Because these penalties depend on coefficient magnitude, the predictor variables are standardized before modeling so that variables measured on different scales are treated comparably. The scaler is fit only on the training data to avoid data leakage.

In [17]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [18]:
def evaluate_model(name, y_true, y_pred):
    return {
        "Model": name,
        "R²": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred)
    }

results = []

In [19]:
# Sportsbook benchmark on the Week 2 test games

results.append(
    evaluate_model(
        "Sportsbook",
        y_test,
        X_test["total"]
    )
)

pd.DataFrame(results)

,Model,R²,MAE,RMSE
0,Sportsbook,0.39481,13.836278,17.767736


## Baseline Linear Regression

In [20]:
linear = LinearRegression()

linear.fit(X_train_scaled, y_train)

linear_pred = linear.predict(X_test_scaled)

results.append(
    evaluate_model(
        "Linear Regression",
        y_test,
        linear_pred
    )
)

## Ridge Regression

In [22]:
ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

ridge_cv = GridSearchCV(
    ridge_pipeline,
    param_grid={
        "ridge__alpha": [0.001, 0.01, 0.1, 1, 10, 100, 1000]
    },
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring="neg_mean_absolute_error"
)

ridge_cv.fit(X_train, y_train)

ridge = ridge_cv.best_estimator_.named_steps["ridge"]
ridge_pred = ridge_cv.predict(X_test)

results.append(
    evaluate_model("Ridge", y_test, ridge_pred)
)

print("Best Ridge alpha:", ridge_cv.best_params_["ridge__alpha"])
print("CV MAE:", -ridge_cv.best_score_)
print("Test MAE:", mean_absolute_error(y_test, ridge_pred))
print("Test RMSE:", root_mean_squared_error(y_test, ridge_pred))

Best Ridge alpha: 0.001
CV MAE: 13.980030852753668
Test MAE: 13.84575935250534
Test RMSE: 17.7692289256107


## Lasso Regression

In [23]:
lasso_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", Lasso(max_iter=20000))
])

lasso_cv = GridSearchCV(
    lasso_pipeline,
    param_grid={
        "lasso__alpha": [0.0001, 0.001, 0.01, 0.1, 1, 10]
    },
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring="neg_mean_absolute_error"
)

lasso_cv.fit(X_train, y_train)

lasso = lasso_cv.best_estimator_.named_steps["lasso"]
lasso_pred = lasso_cv.predict(X_test)

results.append(
    evaluate_model("Lasso", y_test, lasso_pred)
)

print("Best Lasso alpha:", lasso_cv.best_params_["lasso__alpha"])
print("CV MAE:", -lasso_cv.best_score_)
print("Test MAE:", mean_absolute_error(y_test, lasso_pred))
print("Test RMSE:", root_mean_squared_error(y_test, lasso_pred))
print("Lasso coefficients:", lasso.coef_)

Best Lasso alpha: 0.0001
CV MAE: 13.980034055871764
Test MAE: 13.845755971484326
Test RMSE: 17.769225093017784
Lasso coefficients: [13.88138577  0.18664812  0.28054041  0.12460504]


## Elastic Net

In [24]:
elastic_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("elastic", ElasticNet(max_iter=20000))
])

elastic_cv = GridSearchCV(
    elastic_pipeline,
    param_grid={
        "elastic__alpha": [0.0001, 0.001, 0.01, 0.1, 1, 10],
        "elastic__l1_ratio": [0.1, 0.5, 0.9]
    },
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring="neg_mean_absolute_error"
)

elastic_cv.fit(X_train, y_train)

elastic = elastic_cv.best_estimator_.named_steps["elastic"]
elastic_pred = elastic_cv.predict(X_test)

results.append(
    evaluate_model("Elastic Net", y_test, elastic_pred)
)

print("Best parameters:", elastic_cv.best_params_)
print("CV MAE:", -elastic_cv.best_score_)
print("Test MAE:", mean_absolute_error(y_test, elastic_pred))
print("Test RMSE:", root_mean_squared_error(y_test, elastic_pred))
print("Elastic Net coefficients:", elastic.coef_)

Best parameters: {'elastic__alpha': 0.0001, 'elastic__l1_ratio': 0.9}
CV MAE: 13.980034834056472
Test MAE: 13.845758619586256
Test RMSE: 17.76922647967691
Elastic Net coefficients: [13.88125672  0.18667796  0.28059575  0.12464188]


## Results Analysis

In [25]:
results_df = pd.DataFrame(results)

results_df

,Model,R²,MAE,RMSE
0,Sportsbook,0.394810,13.836278,17.767736
1,Linear Regression,0.394708,13.845759,17.769229
2,Ridge,0.394708,13.845759,17.769229
3,Lasso,0.394709,13.845756,17.769225
4,Elastic Net,0.394709,13.845759,17.769226


## Coefficient Comparison

Regularization affects model coefficients differently. Ridge shrinks coefficients toward zero while retaining all predictors. Lasso can shrink coefficients exactly to zero, effectively performing feature selection. Elastic Net combines both Ridge and Lasso penalties.

In [26]:
coef_df = pd.DataFrame({
    "Feature": features,
    "Linear": linear.coef_,
    "Ridge": ridge.coef_,
    "Lasso": lasso.coef_,
    "ElasticNet": elastic.coef_
})

coef_df

,Feature,Linear,Ridge,Lasso,ElasticNet
0,total,13.881493,13.881492,13.881386,13.881257
1,spread,0.186920,0.186920,0.186648,0.186678
2,moneyline_home,0.281313,0.281313,0.280540,0.280596
3,moneyline_away,0.125232,0.125232,0.124605,0.124642


In [27]:
print("Maximum difference between Ridge and Lasso:",
      max(abs(ridge_pred - lasso_pred)))

print("Maximum difference between Ridge and Elastic Net:",
      max(abs(ridge_pred - elastic_pred)))

Maximum difference between Ridge and Lasso: 0.0074707578101538274
Maximum difference between Ridge and Elastic Net: 0.00687892528188172


## Conclusion

Ridge, Lasso, and Elastic Net regression were evaluated to determine whether regularization could improve predictions of NBA game totals. Five-fold cross-validation was used to select the best hyperparameters for each model. All three models selected very small regularization strengths and produced nearly identical results to ordinary linear regression.

The sportsbook benchmark achieved an MAE of 13.8363, while all three regularized models achieved approximately 13.8458. Additionally, Lasso retained all four predictors rather than eliminating any features. These findings suggest that regularization alone does not meaningfully improve predictive accuracy with the current sportsbook variables. Future improvements may require additional features, such as recent team performance, rather than more complex regression techniques.